This notebook was used to fix already captured color-mask photos later utilized to create annotations. It was a one-time correction, and the mistake was already addressed in module environment.py in set_scene_env_for_mask. The issue was that every snapshot was overexposured by setting a parameter ambient intensity to 1, while it should be set to ~0.825. The repair was applied to the 'pure red' pixels, as it was the area of interest (parameter presence area).

Fixed images (spheric color-masks) are available at /IMAGES in DATASET_URL

In [ ]:
import os
from PIL import Image
import numpy as np

def filter_channels(mask_rgb: np.ndarray, channels = "rgb") -> np.ndarray:
    """
    Keep only pixels that are exactly pure channel
    Everything else is set to background [0,0,0].

    """

    M = mask_rgb.astype(np.uint8)
    R = M[:, :, 0]
    G = M[:, :, 1]
    B = M[:, :, 2]

    # yeah, mixing channels was a bad idea, 'pure'
    pure_red = (R > 0) & (G == 0) & (B == 0)
    pure_green = (G > 0) & (B == 0) & (R == 0)
    pure_blue = (B > 0) & (R == 0) & (G == 0)

    out = np.zeros_like(M)

    if "r" in channels:
        out[pure_red, 0] = R[pure_red]  # keep original R id
    # we keep green channel too, as some facades can be extremely wiiide, so it w/h ratio exceeds 2.55
    if "g" in channels:
        out[pure_green, 1] = G[pure_green]

    if "b" in channels:
        out[pure_blue, 2] = B[pure_blue]

    #Image.fromarray(out).show()

    return pure_red


def dim_images(input_dir, output_dir, factor=0.825):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)


    for file_name in os.listdir(input_dir):
        if file_name.lower().startswith(("m-1", "m-2")):

            input_path = os.path.join(input_dir, file_name)
            output_path = os.path.join(output_dir, file_name)

            try:
                with Image.open(input_path) as img:
                    if img.mode in ("RGBA", "P"):
                        img = img.convert("RGB")

                    img_array = np.array(img)
                    #pure_red = filter_channels(img_array, "r")

                    M = img_array.astype(np.uint8)
                    R = M[:, :, 0]
                    G = M[:, :, 1]
                    B = M[:, :, 2]
                    pure_red = (R > 0) & (G == 0) & (B == 0)

                    img_array[pure_red, 0] = np.clip(R[pure_red] * factor, 0, 255).astype(np.uint8)
                    dimmed_img = Image.fromarray(img_array)
                    dimmed_img.save(output_path, quality=100)

            except Exception as e:
                print(f"Error: {file_name}: {e}")



#INPUT_DIR = "C:\\Users\\WA\\Desktop\\misc\\sample_dataset_new"
#OUTPUT_DIR = "C:\\Users\\WA\\Desktop\\misc\\sample_dataset_new_dimmed"

INPUT_DIR = "C:\\Users\\WA\\Desktop\\badanie\\syncity3D\\dataset"
OUTPUT_DIR = "C:\\Users\\WA\\Desktop\\badanie\\syncity3D\\dataset_dimmed"

factor = 0.825

dim_images(INPUT_DIR, OUTPUT_DIR, factor=factor)